# 🌱 PlantTraits2024 - FGVC11 Competition
## EfficientNet-B0 Late Fusion + Log-Scale Label Encoding

> **ÖNEMLİ**: Sağ panelde "Add Input" → "Competition" → "planttraits2024" ekli olmalı!

In [ ]:
!pip install -q timm

In [ ]:
import os, gc, cv2, math, random, warnings
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
import torchvision.transforms as T
import timm

warnings.filterwarnings('ignore')
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## ⚙️ Auto-Detect Data Path & Configuration

In [ ]:
# ============================================================
# AUTO-DETECT DATA PATH - Recursive search
# ============================================================
COMP_DIR = None
TRAIN_IMG_DIR = None
TEST_IMG_DIR = None

def find_train_csv(base):
    if not os.path.exists(base):
        return None
    for root, dirs, files in os.walk(base):
        if 'train.csv' in files:
            return root
    return None

COMP_DIR = find_train_csv('/kaggle/input')

if COMP_DIR:
    print(f'✅ Veri seti bulundu: {COMP_DIR}')
    print(f'   İçerik: {os.listdir(COMP_DIR)}')
    for name in ['train_images', 'train']:
        p = os.path.join(COMP_DIR, name)
        if os.path.exists(p):
            TRAIN_IMG_DIR = p; break
    for name in ['test_images', 'test']:
        p = os.path.join(COMP_DIR, name)
        if os.path.exists(p):
            TEST_IMG_DIR = p; break
    print(f'   Train images: {TRAIN_IMG_DIR}')
    print(f'   Test images: {TEST_IMG_DIR}')
else:
    print('❌ train.csv bulunamadi!')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input','').count(os.sep)
        print(f"  {'  '*level}{os.path.basename(root)}/ {files[:3]}")
        if level > 2: break
    raise FileNotFoundError('Add competition data first!')

In [ ]:
class CFG:
    COMPETITION_DIR = COMP_DIR
    TRAIN_IMAGES = TRAIN_IMG_DIR
    TEST_IMAGES = TEST_IMG_DIR
    OUTPUT_DIR = '/kaggle/working'
    BACKBONE = 'efficientnet_b0'
    IMG_SIZE = 224
    NUM_TARGETS = 6
    TABULAR_DIM = 163
    EPOCHS = 12
    BATCH_SIZE = 32
    LR = 1e-3
    BACKBONE_LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    SEED = 42
    N_FOLDS = 5
    TRAIN_FOLDS = [0, 1, 2, 3, 4]
    R2_LOSS_W = 1.0
    COS_LOSS_W = 0.3
    MSE_LOSS_W = 0.1
    USE_RESPONSE_VAR = True
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

TRAIT_COLS = ['X4_mean','X11_mean','X18_mean','X50_mean','X26_mean','X3112_mean']
TRAIT_SD_COLS = ['X4_sd','X11_sd','X18_sd','X50_sd','X26_sd','X3112_sd']

def seed_everything(seed=42):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)
print(f"Device: {CFG.DEVICE}")

## 🔄 Label Encoder (Log-Scale) — En Kritik Bileşen

In [ ]:
class LabelEncoder:
    def __init__(self):
        self.mean = self.std = None

    def fit(self, y):
        log_y = np.log10(y + 1e-6)
        self.mean, self.std = log_y.mean(axis=0), log_y.std(axis=0)
        print(f"LabelEncoder fitted: mean={np.round(self.mean,3)}, std={np.round(self.std,3)}")
        return self

    def transform(self, y): return (np.log10(y + 1e-6) - self.mean) / self.std
    def inverse_transform(self, y): return 10 ** (y * self.std + self.mean)

    def transform_torch(self, y):
        m = torch.tensor(self.mean, dtype=torch.float32, device=y.device)
        s = torch.tensor(self.std, dtype=torch.float32, device=y.device)
        return (torch.log10(y + 1e-6) - m) / s

    def inverse_transform_torch(self, y):
        m = torch.tensor(self.mean, dtype=torch.float32, device=y.device)
        s = torch.tensor(self.std, dtype=torch.float32, device=y.device)
        return 10 ** (y * s + m)

## 📊 Data Preprocessing

In [ ]:
def preprocess_data(cfg):
    train_df = pd.read_csv(os.path.join(cfg.COMPETITION_DIR, 'train.csv'))
    test_df = pd.read_csv(os.path.join(cfg.COMPETITION_DIR, 'test.csv'))
    print(f"Train: {train_df.shape}, Test: {test_df.shape}")

    # Auto-detect image extension
    sample_id = train_df['id'].iloc[0]
    img_ext = '.jpeg'
    for ext in ['.jpeg', '.jpg', '.png']:
        if os.path.exists(os.path.join(cfg.TRAIN_IMAGES, str(sample_id) + ext)):
            img_ext = ext; break
    print(f"Image extension: {img_ext}")

    train_df['path'] = train_df['id'].apply(lambda x: os.path.join(cfg.TRAIN_IMAGES, str(x) + img_ext))
    test_df['path'] = test_df['id'].apply(lambda x: os.path.join(cfg.TEST_IMAGES, str(x) + img_ext))

    train_df['exists'] = train_df['path'].apply(os.path.exists)
    print(f"Images found: {train_df['exists'].sum()}/{len(train_df)}")
    train_df = train_df[train_df['exists']].reset_index(drop=True)

    if len(train_df) == 0:
        raise ValueError("No images found! Check paths.")

    exclude = set(['id', 'path', 'exists'] + TRAIT_COLS + TRAIT_SD_COLS)
    ancillary_cols = [c for c in train_df.columns
                      if c not in exclude
                      and train_df[c].dtype in ['float64','float32','int64','int32']]
    ancillary_cols = ancillary_cols[:cfg.TABULAR_DIM]
    print(f"Ancillary cols: {len(ancillary_cols)}")

    for col in ancillary_cols:
        med = train_df[col].median()
        train_df[col] = train_df[col].fillna(med)
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna(med)
    for col in ancillary_cols:
        if col not in test_df.columns:
            test_df[col] = 0.0

    for col in TRAIT_COLS:
        if col in train_df.columns:
            lo, hi = train_df[col].quantile(0.01), train_df[col].quantile(0.99)
            train_df[col] = train_df[col].clip(lo, hi)
    for col in TRAIT_COLS + TRAIT_SD_COLS:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(train_df[col].median())
    for col in TRAIT_COLS:
        if col in train_df.columns:
            train_df[col] = train_df[col].clip(lower=1e-6)

    scaler = StandardScaler()
    train_df[ancillary_cols] = scaler.fit_transform(train_df[ancillary_cols].values)
    test_df[ancillary_cols] = scaler.transform(test_df[ancillary_cols].values)
    train_df[ancillary_cols] = train_df[ancillary_cols].replace([np.inf,-np.inf], 0).fillna(0)
    test_df[ancillary_cols] = test_df[ancillary_cols].replace([np.inf,-np.inf], 0).fillna(0)

    le = LabelEncoder()
    le.fit(train_df[TRAIT_COLS].values)

    kf = KFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
    train_df['fold'] = -1
    for fold, (_, vi) in enumerate(kf.split(train_df)):
        train_df.loc[vi, 'fold'] = fold

    return train_df, test_df, ancillary_cols, le, scaler

train_df, test_df, ancillary_cols, label_encoder, scaler = preprocess_data(CFG)
CFG.TABULAR_DIM = len(ancillary_cols)
print(f"Final TABULAR_DIM: {CFG.TABULAR_DIM}")

## 🖼️ Dataset & Transforms

In [ ]:
class PlantDataset(Dataset):
    def __init__(self, df, anc_cols, le=None, transform=None, is_test=False, use_rv=False):
        self.df = df.reset_index(drop=True)
        self.paths = self.df['path'].values
        self.metadata = self.df[anc_cols].values.astype(np.float32)
        self.le = le; self.transform = transform; self.is_test = is_test; self.use_rv = use_rv
        if not is_test:
            self.targets = self.df[TRAIT_COLS].values.astype(np.float32)
            self.targets_sd = self.df[TRAIT_SD_COLS].values.astype(np.float32) if all(c in df.columns for c in TRAIT_SD_COLS) else None

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        try:
            img = cv2.imread(self.paths[idx])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else np.zeros((CFG.IMG_SIZE,CFG.IMG_SIZE,3), dtype=np.uint8)
            img = cv2.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
        except: img = np.zeros((CFG.IMG_SIZE,CFG.IMG_SIZE,3), dtype=np.uint8)
        img = self.transform(img) if self.transform else T.ToTensor()(img)
        tab = torch.nan_to_num(torch.tensor(self.metadata[idx], dtype=torch.float32), nan=0.0)
        if self.is_test: return {'image': img, 'metadata': tab}
        target = self.targets[idx].copy()
        if self.use_rv and self.targets_sd is not None:
            target = np.clip(target + np.random.normal(0, self.targets_sd[idx]), 1e-6, None)
        return {'image': img, 'metadata': tab, 'label': torch.tensor(target, dtype=torch.float32)}

train_tfm = T.Compose([
    T.ToPILImage(), T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.7,1.0)),
    T.RandomHorizontalFlip(0.5), T.RandomVerticalFlip(0.3),
    T.ColorJitter(0.2,0.2,0.2,0.1), T.RandomGrayscale(0.1),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tfm = T.Compose([
    T.ToPILImage(), T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

## 🧠 Model: EfficientNet-B0 + Tabular MLP (Late Fusion)

In [ ]:
class TabularBranch(nn.Module):
    def __init__(self, in_dim=163, hid=128, out=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid), nn.BatchNorm1d(hid), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(hid, hid), nn.BatchNorm1d(hid), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(hid, out))
    def forward(self, x): return self.net(x)

class PlantModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(cfg.BACKBONE, pretrained=True, num_classes=0, global_pool='avg')
        bdim = self.backbone.num_features
        for n, p in self.backbone.named_parameters():
            if any(k in n for k in ['blocks.0','blocks.1','conv_stem','bn1']): p.requires_grad = False
        self.tabular = TabularBranch(cfg.TABULAR_DIM, 128, 128)
        self.head = nn.Sequential(
            nn.Linear(bdim+128, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, cfg.NUM_TARGETS))

    def forward(self, image, metadata):
        return self.head(torch.cat([self.backbone(image), self.tabular(metadata)], dim=1))

_t = PlantModel(CFG)
print(f"Trainable params: {sum(p.numel() for p in _t.parameters() if p.requires_grad):,}")
del _t

## 📉 Losses & Metrics

In [ ]:
class R2Loss(nn.Module):
    def forward(self, pred, true):
        ss_res = ((true - pred)**2).sum(dim=0)
        ss_tot = ((true - true.mean(dim=0))**2).sum(dim=0)
        return (ss_res / (ss_tot + 1e-6)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, r2_w=1.0, cos_w=0.3, mse_w=0.1):
        super().__init__()
        self.r2 = R2Loss(); self.cos = nn.CosineSimilarity(dim=1); self.mse = nn.MSELoss()
        self.r2_w, self.cos_w, self.mse_w = r2_w, cos_w, mse_w
    def forward(self, pred, true):
        return self.r2_w*self.r2(pred,true) + self.cos_w*(1-self.cos(pred,true)).mean() + self.mse_w*self.mse(pred,true)

def compute_r2(yt, yp):
    ss_res = np.sum((yt-yp)**2, axis=0)
    ss_tot = np.sum((yt-yt.mean(axis=0))**2, axis=0)
    r2 = np.clip(1 - ss_res/(ss_tot+1e-6), 0, None)
    return r2.mean(), r2

## 🏋️ Training Loop

In [ ]:
def train_epoch(model, loader, opt, sched, crit, le, dev):
    model.train(); total=0; preds_all=[]; tgts_all=[]
    for batch in tqdm(loader, desc='Train', leave=False):
        img, tab, tgt = batch['image'].to(dev), batch['metadata'].to(dev), batch['label'].to(dev)
        tgt_enc = le.transform_torch(tgt)
        opt.zero_grad()
        out = model(img, tab); loss = crit(out, tgt_enc)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); total += loss.item()
        with torch.no_grad():
            preds_all.append(le.inverse_transform_torch(out).cpu().numpy())
            tgts_all.append(tgt.cpu().numpy())
    r2, per = compute_r2(np.concatenate(tgts_all), np.concatenate(preds_all))
    return total/len(loader), r2, per

@torch.no_grad()
def validate(model, loader, crit, le, dev):
    model.eval(); total=0; preds_all=[]; tgts_all=[]
    for batch in loader:
        img, tab, tgt = batch['image'].to(dev), batch['metadata'].to(dev), batch['label'].to(dev)
        tgt_enc = le.transform_torch(tgt); out = model(img, tab)
        total += crit(out, tgt_enc).item()
        preds_all.append(le.inverse_transform_torch(out).cpu().numpy())
        tgts_all.append(tgt.cpu().numpy())
    r2, per = compute_r2(np.concatenate(tgts_all), np.concatenate(preds_all))
    return total/len(loader), r2, per

@torch.no_grad()
def predict(model, loader, le, dev):
    model.eval(); preds=[]
    for batch in loader:
        out = model(batch['image'].to(dev), batch['metadata'].to(dev))
        preds.append(le.inverse_transform_torch(out).cpu().numpy())
    return np.concatenate(preds)

## 🚀 Main Training Pipeline

In [ ]:
test_ds = PlantDataset(test_df, ancillary_cols, label_encoder, val_tfm, is_test=True)
test_dl = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE*2, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)

all_test_preds = []
all_oof = np.zeros((len(train_df), CFG.NUM_TARGETS))

for fold in CFG.TRAIN_FOLDS:
    print(f"\n{'='*50} FOLD {fold} {'='*50}")
    trn = train_df[train_df['fold']!=fold]; val = train_df[train_df['fold']==fold]
    print(f"Train: {len(trn)}, Val: {len(val)}")

    trn_dl = DataLoader(PlantDataset(trn, ancillary_cols, label_encoder, train_tfm, use_rv=CFG.USE_RESPONSE_VAR),
                        batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_dl = DataLoader(PlantDataset(val, ancillary_cols, label_encoder, val_tfm),
                        batch_size=CFG.BATCH_SIZE*2, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)

    model = PlantModel(CFG).to(CFG.DEVICE)
    bp = [p for n,p in model.named_parameters() if 'backbone' in n and p.requires_grad]
    hp = [p for n,p in model.named_parameters() if 'backbone' not in n and p.requires_grad]
    opt = optim.AdamW([{'params':bp,'lr':CFG.BACKBONE_LR},{'params':hp,'lr':CFG.LR}], weight_decay=CFG.WEIGHT_DECAY)
    sched = OneCycleLR(opt, max_lr=[CFG.BACKBONE_LR, CFG.LR], total_steps=len(trn_dl)*CFG.EPOCHS, pct_start=0.1)
    crit = CombinedLoss(CFG.R2_LOSS_W, CFG.COS_LOSS_W, CFG.MSE_LOSS_W)

    best_r2 = -999
    for ep in range(CFG.EPOCHS):
        tl, tr2, _ = train_epoch(model, trn_dl, opt, sched, crit, label_encoder, CFG.DEVICE)
        vl, vr2, vper = validate(model, val_dl, crit, label_encoder, CFG.DEVICE)
        print(f"  Ep{ep+1:02d} | TrLoss:{tl:.4f} TrR2:{tr2:.4f} | VlLoss:{vl:.4f} VlR2:{vr2:.4f} | {' '.join(f'{v:.3f}' for v in vper)}")
        if vr2 > best_r2:
            best_r2 = vr2; torch.save(model.state_dict(), f'{CFG.OUTPUT_DIR}/best_f{fold}.pth'); print(f"  -> Saved! R2={best_r2:.4f}")

    model.load_state_dict(torch.load(f'{CFG.OUTPUT_DIR}/best_f{fold}.pth'))
    vi = train_df[train_df['fold']==fold].index.values
    oof_dl = DataLoader(PlantDataset(train_df.iloc[vi].reset_index(drop=True), ancillary_cols, label_encoder, val_tfm),
                        batch_size=CFG.BATCH_SIZE*2, shuffle=False, num_workers=CFG.NUM_WORKERS)
    all_oof[vi] = predict(model, oof_dl, label_encoder, CFG.DEVICE)
    all_test_preds.append(predict(model, test_dl, label_encoder, CFG.DEVICE))
    del model, opt, sched; gc.collect(); torch.cuda.empty_cache()

## 📋 OOF Evaluation & Submission

In [ ]:
used = train_df[train_df['fold'].isin(CFG.TRAIN_FOLDS)].index.values
oof_r2, oof_per = compute_r2(train_df.loc[used, TRAIT_COLS].values, all_oof[used])
print(f"\nOOF Mean R2: {oof_r2:.4f}")
for i, c in enumerate(TRAIT_COLS): print(f"  {c}: {oof_per[i]:.4f}")

sub = pd.read_csv(os.path.join(CFG.COMPETITION_DIR, 'sample_submission.csv'))
test_mean = np.mean(all_test_preds, axis=0)
for i, c in enumerate(TRAIT_COLS): sub[c] = np.clip(test_mean[:,i], 1e-6, None)
sub.to_csv(os.path.join(CFG.OUTPUT_DIR, 'submission.csv'), index=False)
print(f"\nSubmission saved: {sub.shape}")
print(sub.head())